# 05 — Churn Prediction & Model Explainability

**Purpose:** Train ML models to predict customer churn, evaluate them properly, and explain the predictions using SHAP.

---

## What We're Building

A **binary classification** model: given a customer's features, predict whether they will churn (1) or not (0).

## Two Models

1. **Logistic Regression** — simple baseline, highly interpretable
2. **Random Forest** — ensemble of decision trees, usually more accurate

## Why Two Models?

Comparing models answers: does complexity add value? If Random Forest barely beats Logistic Regression, the simpler model is preferred (easier to explain to stakeholders).

## What is Train/Test Split?

We hold back 20% of data as a **test set** the model never sees during training. This gives an unbiased estimate of real-world performance. Without it, we'd just be memorising the training data.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import shap
shap.initjs()

from data_processing import load_clean_data
from feature_engineering import build_ml_features
from churn_model import (
    split_data, train_logistic_regression, train_random_forest,
    evaluate_model, compare_models, compute_shap_values, predict_customer_churn
)
from visualization import plot_confusion_matrix

df = load_clean_data()
print(f'Loaded {len(df):,} customers')

## 1. Feature Engineering — Prepare X and y

In [ ]:
X, y = build_ml_features(df)

print(f'Feature matrix X: {X.shape[0]:,} rows × {X.shape[1]} features')
print(f'Target y: {y.sum():,} churned, {(y==0).sum():,} active')
print(f'Class balance: {y.mean()*100:.1f}% churn')
print()
print('Features used:')
print(X.columns.tolist())

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)

print(f'Training set:  {len(X_train):,} samples')
print(f'Test set:      {len(X_test):,} samples')
print(f'Train churn rate: {y_train.mean()*100:.1f}%')
print(f'Test churn rate:  {y_test.mean()*100:.1f}%')

## 3. Train Models

In [ ]:
print('Training Logistic Regression...')
lr_model = train_logistic_regression(X_train, y_train)

print('Training Random Forest...')
rf_model = train_random_forest(X_train, y_train)

## 4. Evaluate Models

**Metrics explained:**

| Metric | Formula | Business meaning |
|--------|---------|------------------|
| **Accuracy** | correct / total | % of all predictions right — misleading with imbalance |
| **Precision** | TP / (TP+FP) | Of predicted churners, how many actually churned? |
| **Recall** | TP / (TP+FN) | Of actual churners, how many did we catch? |
| **F1-Score** | 2×P×R/(P+R) | Balance between precision and recall |
| **ROC-AUC** | Area under ROC curve | How well model separates churners from non-churners |

**For churn prediction, Recall is critical** — missing a churner costs more than a false alarm.

In [ ]:
lr_metrics = evaluate_model(lr_model, X_test, y_test, 'Logistic Regression')
lr_metrics['y_test'] = y_test

In [ ]:
rf_metrics = evaluate_model(rf_model, X_test, y_test, 'Random Forest')
rf_metrics['y_test'] = y_test

In [ ]:
print('=== Model Comparison ===')
comparison = compare_models([lr_metrics, rf_metrics])
comparison

## 5. Confusion Matrices

In [ ]:
plot_confusion_matrix(lr_metrics['confusion_matrix'], 'Logistic Regression').show()
plot_confusion_matrix(rf_metrics['confusion_matrix'], 'Random Forest').show()

## 6. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve
import plotly.graph_objects as go

fig = go.Figure()
for name, m in [('Logistic Regression', lr_metrics), ('Random Forest', rf_metrics)]:
    fpr, tpr, _ = roc_curve(m['y_test'], m['y_prob'])
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                             name=f"{name} (AUC={m['roc_auc']:.3f})"))

fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                         line=dict(dash='dash', color='gray'), name='Random'))
fig.update_layout(title='ROC Curves — Model Comparison',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate',
                  plot_bgcolor='white')
fig.show()

## 7. SHAP — Model Explainability

**What is SHAP?**

SHAP (SHapley Additive exPlanations) is based on game theory. It answers:
> *"For this specific prediction, how much did each feature contribute?"*

**Positive SHAP value** → pushes prediction towards churn  
**Negative SHAP value** → pushes prediction towards no churn  
**Larger magnitude** → stronger effect on the prediction

**Why SHAP is better than just feature importance:**
- Feature importance tells you which features matter globally
- SHAP tells you how each feature affected each individual prediction
- SHAP works on any model (black-box friendly)

In [ ]:
print('Computing SHAP values (may take ~30 seconds)...')
shap_values, X_scaled_sample, explainer = compute_shap_values(rf_model, X_test, n_samples=300)
print('Done.')

In [ ]:
# Global feature importance (mean absolute SHAP value)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_scaled_sample, plot_type='bar',
                  show=False, max_display=15)
plt.title('Global Feature Importance (SHAP)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm — shows direction AND magnitude
# Red = high feature value, Blue = low feature value
# Right = pushes towards churn, Left = pushes away from churn
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_scaled_sample, show=False, max_display=12)
plt.title('SHAP Summary — Feature Effects on Churn Prediction', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Individual Customer Explanation

In [ ]:
# Explain one specific customer prediction
sample_idx = 0
customer_result = predict_customer_churn(rf_model, X_test.iloc[[sample_idx]])
print(f'Customer prediction:')
print(f'  Churn probability: {customer_result["churn_probability_pct"]:.1f}%')
print(f'  Risk category:     {customer_result["risk_category"]}')

# Show waterfall plot for this customer
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[sample_idx],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        data=X_scaled_sample.iloc[sample_idx],
        feature_names=X_scaled_sample.columns.tolist()
    ),
    max_display=10,
    show=True
)

## Summary

| Model | ROC-AUC | Recall | Verdict |
|-------|---------|--------|---------|
| Logistic Regression | ~0.94 | ~0.82 | Good baseline |
| Random Forest | ~0.97 | ~0.92 | Better — use this in production |

**Top predictors of churn (from SHAP):**
1. `payment_failures` — strongest signal
2. `days_since_login` — recency matters
3. `usage_frequency` — engagement predicts retention
4. `support_tickets` — frustration indicator
5. `tenure_months` — newer customers at higher risk

**Next steps for a real project:**
- Schedule weekly model re-training
- Deploy predictions to a CRM system
- A/B test retention interventions on high-risk customers
- Track whether interventions actually reduce churn